# U7-L1：多传感器订阅

**课程目标**

1. 能为一个多路状态观察任务选择订阅方式，用短回调保存各路最近记录，并查看同一观察时刻下的数据值。
2. 能区分源时间戳与本地接收时间，依据缺失、接收后时长及时间戳比较条件判断数据是否适合联合使用。
3. 能比较有限队列的两种丢弃策略，说明选择依据，结束全部订阅，并区分多传感器融合与本课的联合检查。

## 一、任务：为场景运行准备状态观察窗

机器人准备执行一段讲解或踢球程序。操作者需要同时知道：相机有没有画面，机器人是否正在转动，里程计位置有没有变化。每隔一秒显示三路最近记录；某路没有更新时，应明确标出，不能把上一次记录当作刚收到的数据。

本课先做一个不驱动机器人的状态观察窗，再人为放慢图像处理，比较队列如何影响收到的记录。机器人保持固定姿态；可在镜头前缓慢移动彩色卡片观察画面数值变化，无需让机器人运动。

| 需要回答的问题 | 使用的信息 | 程序安排 |
| --- | --- | --- |
| 当前收到的画面是什么样 | 图像尺寸、中心像素、源时间戳 | 图像到达后更新最近记录 |
| 是否有转动读数 | IMU的z轴角速度 | IMU到达后更新数值 |
| 位置和朝向读数是什么 | 里程计x、y、yaw | 里程计到达后更新数值 |
| 这份记录多久没更新 | 本地接收时刻 | 主程序显示时计算经过时长 |

### 1.1 多传感器融合与多模态

**多传感器融合**是把多个传感器的信息结合起来，得到对环境或机器人状态的估计或判断。例如把IMU和里程计的信息用于估计姿态或位置；实际系统还需处理误差、坐标和时间关系。把几路数值打印在一起，只完成了联合观察，不等于已经得到融合定位结果。

**多模态**涉及视觉、语音、动作等不同的信息或表达方式。相机与IMU是不同传感器；视觉检测后播放介绍、执行动作，则涉及多种模态的协作。多传感器融合可以为这种协作提供信息，两者关注的对象不同。

## 二、建立三路订阅

### 2.1 按需读取与订阅

`get_imu()`由主程序主动读取最近可用值；`subscribe_imu(on_imu)`登记一个回调，SDK收到更新时调用它。**回调**就是预先交给SDK、等数据到达时再执行的函数。本课使用SDK已有的回调机制，不需要先创建自己的线程。

回调只提取需要的值，保存为一份新记录；显示、比较和慢处理放在主程序中。这样可减少回调等待。三路分别更新，显示时依次取得的最近记录仍可能来自不同采集时刻。

⚠️ **实验安全**：本课不发运动或模式切换指令。机器人保持坐稳或站稳，避免碰动机器人。若需调整站姿，先按既定模式与人工支撑流程操作，完成后再运行采集。

📌 **前置**：已连接机器人；旧订阅已退订；在Notebook所在目录运行。

**STEP 01：连接并查看模式。**

In [ ]:
import time
from boosteros.robots.booster import BoosterRobot

robot = BoosterRobot()
print("机器人：", robot.robot_info.model, robot.robot_info.serial_number)
print("模式：", robot.get_mode())

### 2.2 同一观察时刻显示哪些值

`latest`中每路保存一份记录，包括数据值、源时间戳和本地接收时刻。`time.monotonic()`用于计算本程序中的时间间隔，不能与SDK源时间戳直接相减。

**STEP 02：订阅三路，每秒显示一次，共观察五次。** 回调每次替换完整记录；主程序取出后使用这份记录，不在同一次打印中反复读取正在更新的字段。`with`覆盖整个观察过程，正常结束或中断时退出并退订。

输出中的“接收后”表示这份记录被本程序接收后经过多久。它不包含传感器采集到回调开始前的传输和排队时间，因此不能直接当作完整延迟。

In [ ]:
latest = {"图像": None, "IMU": None, "里程计": None}

def on_image(image):
    received = time.monotonic()
    pixels = image.to_numpy()
    latest["图像"] = {"source": image.header.timestamp, "received": received,
                      "value": f"{image.width}×{image.height}，中心像素{pixels[image.height//2, image.width//2].tolist()}"}

def on_imu(imu):
    latest["IMU"] = {"source": imu.timestamp, "received": time.monotonic(),
                     "value": f"z轴角速度={imu.angular_velocity[2]:.3f} rad/s"}

def on_odom(odom):
    x, y, yaw = odom.pose_2d
    latest["里程计"] = {"source": odom.timestamp, "received": time.monotonic(),
                        "value": f"x={x:.3f} m，y={y:.3f} m，yaw={yaw:.3f} rad"}

with robot.subscribe_image(on_image, queue_size=1, overflow="drop_oldest"), \
     robot.subscribe_imu(on_imu, queue_size=1, overflow="drop_oldest"), \
     robot.subscribe_odom(on_odom, queue_size=1, overflow="drop_oldest"):
    for number in range(5):
        time.sleep(1.0)
        snapshot = dict(latest)
        observed = time.monotonic()
        print(f"\n观察{number+1}，本地时刻={observed:.3f}")
        for name in ["图像", "IMU", "里程计"]:
            record = snapshot[name]
            if record is None:
                print(name, "尚未收到")
            else:
                age = observed-record["received"]
                print(name, record["value"], "源时间戳=", record["source"],
                      f"接收后{age:.3f}秒")
print("三路观察已结束，订阅已退出")

记录一组完整快照。图像中心像素随卡片变化时，另外两路不一定发生明显变化；读数不变与没有更新要结合接收时刻判断。

| 观察编号 | 图像数值/接收后时长 | IMU数值/接收后时长 | 里程计数值/接收后时长 | 缺失或待查项 |
| --- | --- | --- | --- | --- |
| | | | | |

### 2.3 核对源时间戳

源时间戳描述数据对应的时刻。比较图像和IMU前，须核实单位及计时基准一致；同为一个数值类型不代表可以直接比较。若单位为秒且同源，图像－IMU为负，表示这份图像的时刻较早。各路最近记录的差值没有完成同步配对，也不表示当前数据年龄。

**STEP 03：检查最后一份快照的时间戳。** 未核实时保持`False`，记录原值即可。

In [ ]:
TIMESTAMPS_VERIFIED = False  # 教师核实单位为秒且基准一致后修改
if not TIMESTAMPS_VERIFIED:
    print("本次仅保留源时间戳原值，单位与基准待核")
elif snapshot["图像"] is None or snapshot["IMU"] is None or snapshot["里程计"] is None:
    print("缺少记录，不能比较")
else:
    print("图像－IMU：", (snapshot["图像"]["source"]-snapshot["IMU"]["source"])*1000, "毫秒")
    print("图像－里程计：", (snapshot["图像"]["source"]-snapshot["里程计"]["source"])*1000, "毫秒")

## 三、处理跟不上时保留什么

队列保存等待回调处理的数据。传感器订阅`queue_size=0`表示不设容量上限；本课状态观察需要近期数据，因此显式设置有限容量。

| 队列满时的策略 | 处理 | 适用取舍 |
| --- | --- | --- |
| drop_oldest | 丢弃最旧的待处理项，留下新到的数据 | 优先了解近期状态，但会遗漏中间数据 |
| drop_newest | 丢弃新到的数据，保留正在等待的旧项 | 保留已排队项，收到的记录可能更旧 |
| block | 等待队列有空位 | 会产生等待，不能保证完整链路无损 |

### 3.1 用慢回调比较两种策略

**STEP 04：运行下面定义。** 这里特意把图像回调放慢到约0.3秒，用于暴露排队现象；实际状态观察应采用STEP 02的短回调。实验保持机器人和场景条件相近，分别记录两种策略。回调中只采集时间记录，实验结束后打印，减少打印对比较的干扰。

In [ ]:
def queue_trial(policy):
    rows = []
    def slow_image(image):
        received = time.monotonic()
        time.sleep(0.3)
        rows.append((image.header.timestamp, received))
    with robot.subscribe_image(slow_image, queue_size=1, overflow=policy):
        time.sleep(3.0)
    print("策略：", policy, "已处理条数：", len(rows))
    for stamp, received in rows[-5:]:
        print("源时间戳：", stamp, "回调开始的本地时刻：", round(received, 3))
    return rows

**STEP 05：先运行drop_oldest，结束后再运行下一格。** 运行期间可以缓慢移动卡片，不碰机器人。

In [ ]:
oldest_rows = queue_trial("drop_oldest")

**STEP 06：比较drop_newest。** 条数接近不表示保留了相同的数据。核实时间戳单位后，可比较相邻源时刻间隔与回调开始间隔；本实验没有记录全部原始消息，因此不能准确统计每条被丢弃的数据，也不能保证每次差异都明显。

In [ ]:
newest_rows = queue_trial("drop_newest")

### 3.2 为状态观察窗作选择

| 策略 | 已处理条数 | 末几条源时间戳与间隔 | 本次能确认什么 | 仍不能确认什么 |
| --- | --- | --- | --- | --- |
| drop_oldest | | | | |
| drop_newest | | | | |

状态观察窗关心近期读数，可以选择小队列和`drop_oldest`，同时显示接收后时长；若要保存全部采样用于离线分析，还需另行设计处理能力和存储，不能把本课设置当作无损采集方案。

本课实验结束时，各格的`with`均已退出，无需切换模式来证明退订。按教师安排保持当前安全姿态；需要关机时执行既定人工收尾流程。

## 四、学习检查

1. 在STEP 02某次观察中，三路数值分别是什么？哪路记录被接收得更早？这一结果能否说明采集顺序？
2. 某路“接收后”很短，但源数据已经排队很久，显示值能证明端到端延迟很小吗？说明还缺少什么信息。
3. 为当前状态窗选择队列策略，并结合本次记录解释代价；若两次实验差异不明显，应如实记录。
4. 说明多传感器融合、多路联合打印和视觉—语音—动作协作各关注什么。给出一个需要融合估计的例子，但不把本实验称为已实现定位融合。

**完成核对**：有数值的三路快照、源时间戳核实状态、队列记录和策略说明均已保存；全部订阅已退出。